# ============================================================
# EDA CCAA CANARIAS — calima_mortality
# Escale: Autonomus Community (total aggregation Canarias)
# ============================================================

# 1. IMPORTS


In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
from pathlib import Path

ROOT = Path(r'C:\Users\fdora\RA_Career\Projects\climate_mortality')
sys.path.insert(0, str(ROOT))

from src.utils.d25_nb_utils import (
    autosave_fig, save_table,
)
REPORTS_DIR = ROOT / "reports" / "ccaa"
TAB_DIR = REPORTS_DIR / "tables" 
TAB_DIR.mkdir(parents=True, exist_ok=True)
FP = ROOT / "data/processed/provinces/master_ccaa_canarias_2016_2025.parquet"
pd.set_option('display.max_columns', None)


# 2. LOAD DATA


In [41]:
# C:\Users\fdora\RA_Career\Projects\climate_mortality\data\processed\provinces\master_ccaa_canarias_2016_2025.parquet
df = pd.read_parquet(FP)

print(df.shape)
print(df.dtypes)
print(df.head())



(522, 12)
week_start           datetime64[ns]
province                     object
population                    int64
deaths                        int64
deaths_per_100k             float64
calima_sct                     bool
calima_lp                      bool
calima_any                     bool
calima_both                    bool
pop_intense                   int64
pct_exposed_ccaa            float64
calima_level_ccaa            object
dtype: object
  week_start  province  population  deaths  deaths_per_100k  calima_sct  \
0 2016-01-04  canarias     2155000     299        13.874710       False   
1 2016-01-11  canarias     2155000     340        15.777262       False   
2 2016-01-18  canarias     2155000     329        15.266821       False   
3 2016-01-25  canarias     2155000     291        13.503480       False   
4 2016-02-01  canarias     2155000     302        14.013921       False   

   calima_lp  calima_any  calima_both  pop_intense  pct_exposed_ccaa  \
0      False       Fa

# 3. QUICK QA


In [42]:
print(df.isnull().sum())
print(df['calima_level_ccaa'].value_counts())  # ajusta nombre si difiere




week_start           0
province             0
population           0
deaths               0
deaths_per_100k      0
calima_sct           0
calima_lp            0
calima_any           0
calima_both          0
pop_intense          0
pct_exposed_ccaa     0
calima_level_ccaa    0
dtype: int64
calima_level_ccaa
no_calima    464
intense       37
possible      18
probable       3
Name: count, dtype: int64


# 4. DESCRIPTION — deaths by calima level


In [43]:
summary = df.groupby('calima_level_ccaa')['deaths'].agg(['mean', 'median', 'std', 'count'])
print(summary)



                         mean  median        std  count
calima_level_ccaa                                      
intense            359.702703   353.0  47.327972     37
no_calima          319.795259   316.5  43.905296    464
possible           342.055556   347.5  44.935866     18
probable           367.333333   339.0  64.360961      3


# 5. ANOVA


In [44]:
groups = [group['deaths'].values for _, group in df.groupby('calima_level_ccaa')]
f_stat, p_value = stats.f_oneway(*groups)
print(f'F-statistic: {f_stat:.4f}')
print(f'P-value: {p_value:.4f}')

# η² manual
ss_between = sum(
    len(g) * (g['deaths'].mean() - df['deaths'].mean())**2
    for _, g in df.groupby('calima_level_ccaa')
)
ss_total = ((df['deaths'] - df['deaths'].mean())**2).sum()
eta_sq = ss_between / ss_total
print(f'η² CCAA: {eta_sq:.4f}')



F-statistic: 11.3567
P-value: 0.0000
η² CCAA: 0.0617


# 6. PAIRWISE T-TESTS


In [45]:
#pairwise = pg.pairwise_tests(data=df, dv='deaths_total', between='calima_level_ccaa', padjust='bonf')
#print(pairwise)
from itertools import combinations

levels = df['calima_level_ccaa'].unique()
for a, b in combinations(sorted(levels), 2):
    g1 = df[df['calima_level_ccaa'] == a]['deaths']
    g2 = df[df['calima_level_ccaa'] == b]['deaths']
    t, p = stats.ttest_ind(g1, g2)
    print(f'{a} vs {b}: t={t:.3f}, p={p:.4f}')



intense vs no_calima: t=5.290, p=0.0000
intense vs possible: t=1.319, p=0.1930
intense vs probable: t=-0.263, p=0.7941
no_calima vs possible: t=-2.109, p=0.0355
no_calima vs probable: t=-1.865, p=0.0628
possible vs probable: t=-0.856, p=0.4027


# 7. LAG ANALYSIS


In [46]:
for lag in [0, 1, 2]:
    df[f'deaths_lag{lag}'] = df['deaths'].shift(lag)

for lag in [0, 1, 2]:
    mask = df[f'deaths_lag{lag}'].notna()
    corr, pval = stats.pearsonr(
        df.loc[mask, 'pct_exposed_ccaa'],
        df.loc[mask, f'deaths_lag{lag}']
    )
    print(f'Lag{lag}: r={corr:.4f}, p={pval:.4f}')



Lag0: r=0.2534, p=0.0000
Lag1: r=0.2138, p=0.0000
Lag2: r=0.1986, p=0.0000


# 8. TABLE η² MULTILEVEL — 5 escales


In [47]:
# Load η² from saved ANOVA tables
ref_tfe  = pd.read_csv(r'C:\Users\fdora\RA_Career\Projects\climate_mortality\reports\islands\tables\tenerife\anova_insular_tfe.csv')
ref_gcan = pd.read_csv(r'C:\Users\fdora\RA_Career\Projects\climate_mortality\reports\islands\tables\gran_canaria\anova_insular_gcan.csv')
ref_sct  = pd.read_csv(r'C:\Users\fdora\RA_Career\Projects\climate_mortality\reports\provinces\tables\anova_provincial_sct.csv')
ref_lp   = pd.read_csv(r'C:\Users\fdora\RA_Career\Projects\climate_mortality\reports\provinces\tables\anova_provincial_lp.csv')

eta_table = pd.DataFrame({
    'escala':  ['TFE insular', 'GC insular', 'SC Tenerife provincial', 'Las Palmas provincial', 'Canarias CCAA'],
    'nivel':   ['island', 'island', 'provincial', 'provincial', 'ccaa'],
    'eta_sq':  [
        ref_tfe.loc[0,  'eta_squared'],
        ref_gcan.loc[0, 'eta_squared'],
        ref_sct.loc[0,  'eta_squared'],
        ref_lp.loc[0,   'eta_squared'],
        eta_sq
    ]
})

print(eta_table.sort_values('eta_sq', ascending=False).to_string(index=False))

save_table(eta_table, TAB_DIR, filename="eta2_multinivel.csv", index=False)

                escala      nivel   eta_sq
         Canarias CCAA       ccaa 0.061714
            GC insular     island 0.057577
SC Tenerife provincial provincial 0.056334
           TFE insular     island 0.054066
 Las Palmas provincial provincial 0.049512
Saved table -> C:\Users\fdora\RA_Career\Projects\climate_mortality\reports\ccaa\tables\eta2_multinivel.csv


WindowsPath('C:/Users/fdora/RA_Career/Projects/climate_mortality/reports/ccaa/tables/eta2_multinivel.csv')